# Dantzig and Ramser's Truck Dispatching Problem

In 1959, George Dantzig and John Ramser introduced the **Truck Dispatching Problem**, now recognized as an early capacitated vehicle-routing problem (CVRP). A fleet delivers gasoline from terminal $P_0$ to stations $P_1,\ldots,P_{12}$. Each station must be visited exactly once, each trip begins and ends at the terminal, and the total demand on a trip cannot exceed the truck capacity $C=6000$.

For routes $R$, the objective is to minimize total mileage

$$\min \sum_{r\in R}\sum_{(i,j)\in r} d_{ij},$$

subject to the capacity condition

$$\sum_{i\in r} q_i \le C \qquad \text{for every route }r.$$

The paper's two-stage aggregation procedure produced a near-optimal solution of **294** distance units. Dantzig and Ramser conjectured that a different assignment totaling **290** was the true optimum, but did not prove that claim in the paper.

## Published Table 1 Instance

The terminal is $P_0$ and the twelve station demands are:

| Station | P1 | P2 | P3 | P4 | P5 | P6 | P7 | P8 | P9 | P10 | P11 | P12 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Demand | 1200 | 1700 | 1500 | 1400 | 1700 | 1400 | 1200 | 1900 | 1800 | 1600 | 1700 | 1100 |

The complete symmetric distance matrix transcribed from Table 1 is:

| | P0 | P1 | P2 | P3 | P4 | P5 | P6 | P7 | P8 | P9 | P10 | P11 | P12 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| **P0** | 0 | 9 | 14 | 21 | 23 | 22 | 25 | 32 | 36 | 38 | 42 | 50 | 52 |
| **P1** | 9 | 0 | 5 | 12 | 22 | 21 | 24 | 31 | 35 | 37 | 41 | 49 | 51 |
| **P2** | 14 | 5 | 0 | 7 | 17 | 16 | 23 | 26 | 30 | 36 | 36 | 44 | 46 |
| **P3** | 21 | 12 | 7 | 0 | 10 | 21 | 30 | 27 | 37 | 43 | 31 | 37 | 39 |
| **P4** | 23 | 22 | 17 | 10 | 0 | 19 | 28 | 25 | 35 | 41 | 29 | 31 | 29 |
| **P5** | 22 | 21 | 16 | 21 | 19 | 0 | 9 | 10 | 16 | 22 | 20 | 28 | 30 |
| **P6** | 25 | 24 | 23 | 30 | 28 | 9 | 0 | 7 | 11 | 13 | 17 | 25 | 27 |
| **P7** | 32 | 31 | 26 | 27 | 25 | 10 | 7 | 0 | 10 | 16 | 10 | 18 | 20 |
| **P8** | 36 | 35 | 30 | 37 | 35 | 16 | 11 | 10 | 0 | 6 | 6 | 14 | 16 |
| **P9** | 38 | 37 | 36 | 43 | 41 | 22 | 13 | 16 | 6 | 0 | 12 | 12 | 20 |
| **P10** | 42 | 41 | 36 | 31 | 29 | 20 | 17 | 10 | 6 | 12 | 0 | 17 | 10 |
| **P11** | 50 | 49 | 44 | 37 | 31 | 28 | 25 | 18 | 14 | 12 | 17 | 0 | 10 |
| **P12** | 52 | 51 | 46 | 39 | 29 | 30 | 27 | 20 | 16 | 20 | 10 | 10 | 0 |

## Reproducing the Python Module

The reproduced implementation follows the paper's two aggregation stages:

1. Stage 1 pairs stations only when their combined demand is at most $C/2=3000$.
2. Stage 2 pairs those aggregates when their combined demand is at most $C=6000$.

The historical paper solved pairing relaxations and manually resolved fractional solutions. The Python reproduction instead uses exact integer maximum-weight matching at each stage. It then performs a separate exhaustive set-partitioning search over all capacity-feasible customer subsets to test the paper's conjecture.

In [10]:
import itertools

# Dantzig & Ramser (1959) - Table 1 Data
# 12 delivery points (P1 to P12) and 1 terminal (P0)
CAPACITY = 6000
DEMANDS = [0, 1200, 1700, 1500, 1400, 1700, 1400, 1200, 1900, 1800, 1600, 1700, 1100]

# Lower triangular array distances extracted from Table 1
LOWER_TRIANGULAR = [
    [],                                         # P0 (Terminal)
    [9],                                        # P1
    [14, 5],                                    # P2
    [21, 12, 7],                                # P3
    [23, 22, 17, 10],                           # P4
    [22, 21, 16, 21, 19],                       # P5
    [25, 24, 23, 30, 28, 9],                    # P6
    [32, 31, 26, 27, 25, 10, 7],                # P7
    [36, 35, 30, 37, 35, 16, 11, 10],           # P8
    [38, 37, 36, 43, 41, 22, 13, 16, 6],        # P9
    [42, 41, 36, 31, 29, 20, 17, 10, 6, 12],    # P10
    [50, 49, 44, 37, 31, 28, 25, 18, 14, 12, 17],# P11
    [52, 51, 46, 39, 29, 30, 27, 20, 16, 20, 10, 10] # P12
]

# Build the symmetric 13x13 distance matrix
D = [[0]*13 for _ in range(13)]
for i in range(13):
    for j in range(len(LOWER_TRIANGULAR[i])):
        D[i][j] = LOWER_TRIANGULAR[i][j]
        D[j][i] = LOWER_TRIANGULAR[i][j]


def get_min_loop(points):
    """
    Finds the exact minimum TSP loop distance starting from Terminal (P0),
    visiting all given points, and returning to the Terminal.
    Uses exhaustive permutations (safe as len(points) <= 4 in this algorithm).
    """
    if not points:
        return 0
    min_dist = float('inf')
    best_path = None
    for p in itertools.permutations(points):
        # Start at P0 -> first point
        dist = D[0][p[0]]
        # Travel through the points
        for i in range(len(p) - 1):
            dist += D[p[i]][p[i+1]]
        # Return to P0
        dist += D[p[-1]][0]
        if dist < min_dist:
            min_dist = dist
            best_path = p
    return min_dist, best_path


def get_max_weight_matching(elements, weights_dict):
    """
    Exact Integer Maximum Weight Matching.
    Replaces the manual linear-programming & fraction-resolution steps 
    detailed in the paper with an exact subset-backtracking algorithm.
    """
    best_weight = -1
    best_matching = []

    def backtrack(remaining, current_weight, current_matching):
        nonlocal best_weight, best_matching
        if not remaining:
            if current_weight > best_weight:
                best_weight = current_weight
                best_matching = current_matching[:]
            return

        # Pop the first element to process it (prevents symmetrical duplicate trees)
        v = remaining[0]

        # Option 1: v remains an unpaired singleton
        backtrack(remaining[1:], current_weight, current_matching + [[v]])

        # Option 2: v is paired with some available u
        for i in range(1, len(remaining)):
            u = remaining[i]
            pair = tuple(sorted((v, u)))
            if pair in weights_dict:
                # Remove u from the remaining pool
                new_remaining = remaining[1:i] + remaining[i+1:]
                backtrack(new_remaining, current_weight + weights_dict[pair], current_matching + [[v, u]])

    backtrack(elements, 0, [])
    return best_matching, best_weight


def main():
    print("=" * 70)
    print(" THE TRUCK DISPATCHING PROBLEM - Dantzig & Ramser (1959)")
    print(" Sample Instance from Table 1 Reimplementation")
    print("=" * 70)

    # =========================================================
    # STAGE 1 AGGREGATION
    # Condition: Pairs allowed if combined demand <= C/2 = 3000
    # =========================================================
    print("\n--- STAGE 1 AGGREGATION (Capacity limit: 3000) ---")
    weights_stage1 = {}
    for i in range(1, 13):
        for j in range(i + 1, 13):
            if DEMANDS[i] + DEMANDS[j] <= CAPACITY // 2:
                # Savings calculation exactly as defined in the paper's delta-function
                saving = D[0][i] + D[0][j] - D[i][j]
                weights_stage1[(i, j)] = saving

    s1_elements = list(range(1, 13))
    s1_matching, s1_saving = get_max_weight_matching(s1_elements, weights_stage1)
    
    # Sort aggregates for deterministic, neat output
    s1_matching = sorted([sorted(group) for group in s1_matching], key=lambda x: x[0])
    
    stage1_loops = []
    print("Optimal Stage 1 Aggregates (Yielding Max Savings):")
    for idx, agg in enumerate(s1_matching):
        dem = sum(DEMANDS[p] for p in agg)
        cost, path = get_min_loop(agg)
        stage1_loops.append(cost)
        print(f"  A{idx+1}: {agg}".ljust(20) + f" | Demand: {dem}".ljust(18) + f" | Loop Dist: {cost}")
    print(f"Stage 1 Total Savings: {s1_saving}")

    # =========================================================
    # STAGE 2 AGGREGATION
    # Condition: Aggregate pairs allowed if combined demand <= 6000
    # =========================================================
    print("\n--- STAGE 2 AGGREGATION (Capacity limit: 6000) ---")
    weights_stage2 = {}
    for i in range(len(s1_matching)):
        for j in range(i + 1, len(s1_matching)):
            dem_i = sum(DEMANDS[v] for v in s1_matching[i])
            dem_j = sum(DEMANDS[v] for v in s1_matching[j])
            
            if dem_i + dem_j <= CAPACITY:
                combined_pts = s1_matching[i] + s1_matching[j]
                cost_ij, _ = get_min_loop(combined_pts)
                
                # Savings mapping aggregate pairs against their independent loops
                saving = stage1_loops[i] + stage1_loops[j] - cost_ij
                weights_stage2[(i, j)] = saving

    s2_elements = list(range(len(s1_matching)))
    s2_matching, s2_saving = get_max_weight_matching(s2_elements, weights_stage2)

    # Resolve Final Routes
    print("\n--- FINAL DANTZIG-RAMSER SOLUTION ---")
    dantzig_ramser_cost = 0
    final_dr_routes = []
    for group in s2_matching:
        route_pts = []
        for idx in group:
            route_pts.extend(s1_matching[idx])
        cost, path = get_min_loop(route_pts)
        dantzig_ramser_cost += cost
        final_dr_routes.append((list(path), cost, sum(DEMANDS[p] for p in route_pts)))

    # Sort routes by length
    final_dr_routes.sort(key=lambda x: len(x[0]), reverse=True)
    
    for idx, (path, cost, dem) in enumerate(final_dr_routes):
        fmt_path = " -> ".join([f"P{p}" for p in path])
        print(f"Trip {idx+1}: Terminal -> {fmt_path} -> Terminal")
        print(f"        Demand: {dem} | Distance: {cost}")

    print(f"\n>> Total Distance (Dantzig-Ramser Method): {dantzig_ramser_cost} units")
    print("*(Matches the paper's 'best solution' result of 294 exactly)*")


    # =========================================================
    # EXACT OPTIMAL VRP SOLVER (Set Partitioning verification)
    # Generates all valid subset capacities and exacts best cover
    # =========================================================
    print("\n" + "=" * 70)
    print(" EXACT VRP OPTIMAL SOLUTION (Verification against Paper Conjecture)")
    print("=" * 70)
    
    valid_subsets = []
    subset_costs = {}
    
    for mask in range(1, 1 << 12):
        pts = []
        dem = 0
        for bit in range(12):
            if mask & (1 << bit):
                pts.append(bit + 1)
                dem += DEMANDS[bit + 1]
                
        if dem <= CAPACITY:
            cost, path = get_min_loop(pts)
            valid_subsets.append(tuple(pts))
            subset_costs[tuple(pts)] = (cost, path)

    best_vrp_cost = float('inf')
    best_vrp_routes = []

    def solve_vrp(remaining_bits, current_cost, current_routes):
        nonlocal best_vrp_cost, best_vrp_routes
        if current_cost >= best_vrp_cost:
            return # Branch pruning (Bounds)
        if remaining_bits == 0:
            best_vrp_cost = current_cost
            best_vrp_routes = current_routes[:]
            return
            
        first_bit = 0
        while not (remaining_bits & (1 << first_bit)):
            first_bit += 1
            
        for pts in valid_subsets:
            if (first_bit + 1) in pts:
                can_use = True
                for p in pts:
                    if not (remaining_bits & (1 << (p - 1))):
                        can_use = False
                        break
                if can_use:
                    new_bits = remaining_bits
                    for p in pts:
                        new_bits &= ~(1 << (p - 1))
                    solve_vrp(new_bits, current_cost + subset_costs[pts][0], current_routes + [pts])

    solve_vrp((1 << 12) - 1, 0, [])

    best_vrp_routes.sort(key=lambda x: len(x), reverse=True)
    for idx, r_pts in enumerate(best_vrp_routes):
        cost, path = subset_costs[r_pts]
        dem = sum(DEMANDS[p] for p in r_pts)
        fmt_path = " -> ".join([f"P{p}" for p in path])
        print(f"Optimal Trip {idx+1}: Terminal -> {fmt_path} -> Terminal")
        print(f"                Demand: {dem} | Distance: {cost}")

    print(f"\n>> Total Distance (True Exact Optimum): {best_vrp_cost} units")
    print("*(Matches the paper's conjectured true optimum of 290 exactly!)*")
    print("=" * 70)


if __name__ == "__main__":
    main()

 THE TRUCK DISPATCHING PROBLEM - Dantzig & Ramser (1959)
 Sample Instance from Table 1 Reimplementation

--- STAGE 1 AGGREGATION (Capacity limit: 3000) ---
Optimal Stage 1 Aggregates (Yielding Max Savings):
  A1: [1, 2]         | Demand: 2900    | Loop Dist: 28
  A2: [3, 4]         | Demand: 2900    | Loop Dist: 54
  A3: [5]            | Demand: 1700    | Loop Dist: 44
  A4: [6, 10]        | Demand: 3000    | Loop Dist: 84
  A5: [7, 9]         | Demand: 3000    | Loop Dist: 86
  A6: [8]            | Demand: 1900    | Loop Dist: 72
  A7: [11, 12]       | Demand: 2800    | Loop Dist: 112
Stage 1 Total Savings: 248

--- STAGE 2 AGGREGATION (Capacity limit: 6000) ---

--- FINAL DANTZIG-RAMSER SOLUTION ---
Trip 1: Terminal -> P1 -> P2 -> P3 -> P4 -> Terminal
        Demand: 5800 | Distance: 54
Trip 2: Terminal -> P7 -> P12 -> P11 -> P9 -> Terminal
        Demand: 5800 | Distance: 112
Trip 3: Terminal -> P6 -> P10 -> P8 -> Terminal
        Demand: 4900 | Distance: 84
Trip 4: Terminal -> P5 -

The **294-unit** result reproduces the paper's aggregation outcome. The subsequent **290-unit** result comes from an independent exact set-partitioning enumeration; it is not part of the historical aggregation algorithm. Because every capacity-feasible customer subset is considered and each subset's shortest terminal loop is computed exactly, this finite search proves the paper's conjecture for the published instance.

## Exact CVRP Formulation in OPL

The OPL model uses binary directed arc variables $x_{ij}$. Every customer has exactly one incoming and one outgoing arc, while depot departures and returns are balanced. Continuous cumulative-load variables satisfy

$$u_j \ge u_i + q_j - C(1-x_{ij}),$$

for customer-to-customer arcs. These MTZ-style constraints enforce route capacity and eliminate customer-only subtours because load must strictly increase along every selected customer arc.

Unlike the paper's pairing relaxation, this is an integral CVRP formulation whose optimum can directly verify the 290-unit conjecture.

In [11]:
model_text = '''
/*
  Exact capacitated vehicle-routing formulation of the Truck Dispatching Problem.

  The terminal is node 0 and customer stations are nodes 1..N. Each customer is
  visited exactly once, every route starts and ends at the terminal, and the
  cumulative delivery load on each route cannot exceed truck capacity.

  This is an exact directed CVRP model, not the paper's original undirected
  pairing/aggregation LP relaxation. The binary arc variables enforce integral
  routes directly.
*/

// Number of customer stations and node index sets.
int N = ...;
range Customers = 1..N;
range Nodes = 0..N;

// Distance between nodes, customer demands, and common truck capacity.
param float distance[Nodes][Nodes];
param int demand[Customers];
param int capacity;

// x[i][j] is 1 when a route travels directly from node i to node j.
dvar boolean x[Nodes][Nodes];

// load[i] is the cumulative delivery load after serving customer i.
dvar float+ load[Customers];

// Minimize the total mileage of all selected route legs.
minimize totalMileage:
  sum(i in Nodes, j in Nodes: i != j)
    distance[i][j] * x[i][j];

subject to {
  // Self-loops do not represent valid route legs.
  forall(i in Nodes)
    noSelfLoop: x[i][i] == 0;

  // Each customer has exactly one outgoing route leg.
  forall(i in Customers)
    stationDeparture:
      sum(j in Nodes: j != i) x[i][j] == 1;

  // Each customer has exactly one incoming route leg.
  forall(j in Customers)
    stationArrival:
      sum(i in Nodes: i != j) x[i][j] == 1;

  // The number of routes leaving the terminal equals the number returning.
  depotFlowBalance:
    sum(j in Customers) x[0][j]
    == sum(i in Customers) x[i][0];

  // The load after serving a station must cover that station's demand.
  forall(i in Customers)
    minimumStationLoad: load[i] >= demand[i];

  // No route load may exceed truck capacity.
  forall(i in Customers)
    truckCapacity: load[i] <= capacity;

  // A selected customer-to-customer leg adds the next station's demand;
  // the capacity term relaxes the inequality when the leg is not selected.
  forall(i in Customers, j in Customers: i != j)
    loadPropagation:
      load[j] >= load[i] + demand[j] - capacity * (1 - x[i][j]);
}
'''

data_text = '''
/*
  Published twelve-station truck-dispatching instance.
  Node 0 is the bulk terminal; nodes 1..12 are customer stations.
  Demands are in gallons and distances are mileage units.

  The distance matrix and demand vector are transcribed from Table 1.
*/

N = 12;
capacity = 6000;

// Station demands q[i] for customer stations 1..12.
demand = [
  1200,
  1700,
  1500,
  1400,
  1700,
  1400,
  1200,
  1900,
  1800,
  1600,
  1700,
  1100
];

// Symmetric distance matrix for nodes 0..12.
distance = [
  [0, 9, 14, 21, 23, 22, 25, 32, 36, 38, 42, 50, 52],
  [9, 0, 5, 12, 22, 21, 24, 31, 35, 37, 41, 49, 51],
  [14, 5, 0, 7, 17, 16, 23, 26, 30, 36, 36, 44, 46],
  [21, 12, 7, 0, 10, 21, 30, 27, 37, 43, 31, 37, 39],
  [23, 22, 17, 10, 0, 19, 28, 25, 35, 41, 29, 31, 29],
  [22, 21, 16, 21, 19, 0, 9, 10, 16, 22, 20, 28, 30],
  [25, 24, 23, 30, 28, 9, 0, 7, 11, 13, 17, 25, 27],
  [32, 31, 26, 27, 25, 10, 7, 0, 10, 16, 10, 18, 20],
  [36, 35, 30, 37, 35, 16, 11, 10, 0, 6, 6, 14, 16],
  [38, 37, 36, 43, 41, 22, 13, 16, 6, 0, 12, 12, 20],
  [42, 41, 36, 31, 29, 20, 17, 10, 6, 12, 0, 17, 10],
  [50, 49, 44, 37, 31, 28, 25, 18, 14, 12, 17, 0, 10],
  [52, 51, 46, 39, 29, 30, 27, 20, 16, 20, 10, 10, 0]
];
'''

## Solving with PyOPL and HiGHS

PyOPL compiles the OPL model and delegates the mixed-integer program to HiGHs.

In [ ]:
%%capture
! pip install rhetor

In [12]:
from pyopl import solve

tdp_file_model = "tdp.mod"
tdp_file_data = "tdp.dat"

with open(tdp_file_model, "w") as f:
    f.write(model_text)

with open(tdp_file_data, "w") as f:
    f.write(data_text)

results = solve(str(tdp_file_model), str(tdp_file_data), solver="scipy")
stats = results.get("stats", {})

print(f"Status: {results['status']}")
print(f"Objective: {results['objective_value']:.0f} distance units")
print(f"MIP gap: {stats.get('MIPGap', float('nan')):.2%}")
print(f"Runtime: {stats.get('Runtime', float('nan')):.2f} seconds")

assert results["status"] == "OPTIMAL"
assert abs(results["objective_value"] - 290) < 1e-6

PyOPL: compiling model...
PyOPL: compilation complete; starting SciPy/HiGHS optimization.
PyOPL/SciPy-HiGHS: variables=181, equalities=38, inequalities=156, integrality=169
Running HiGHS 1.12.0 (git hash: 4f96ee8): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 194 rows; 181 cols; 745 nonzeros; 169 integer variables (156 binary)
Coefficient ranges:
  Matrix  [1e+00, 6e+03]
  Cost    [5e+00, 5e+01]
  Bound   [1e+00, 6e+03]
  RHS     [1e+00, 6e+03]
Presolving model
157 rows, 168 cols, 708 nonzeros  0s
157 rows, 168 cols, 708 nonzeros  0s
Presolve reductions: rows 157(-37); columns 168(-13); nonzeros 708(-37) 
Objective function is integral with scale 1

Solving MIP model with:
   157 rows
   168 cols (156 binary, 0 integer, 0 implied int., 12 continuous, 0 domain fixed)
   708 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => So

## Extract and Validate the Dispatch Plan

In [13]:
import re

arc_pattern = re.compile(r"x(?:\[(\d+),(\d+)\]|_(\d+)_(\d+))")


def parse_arc(variable_name):
    match = arc_pattern.fullmatch(variable_name)
    if not match:
        return None
    indices = match.group(1, 2) if match.group(1) is not None else match.group(3, 4)
    return tuple(map(int, indices))


selected_arcs = []
for variable_name, value in results["solution"].items():
    arc = parse_arc(variable_name)
    if arc is not None and value > 0.5:
        selected_arcs.append(arc)

successor = dict(selected_arcs)
demands = [0, 1200, 1700, 1500, 1400, 1700, 1400, 1200, 1900, 1800, 1600, 1700, 1100]
distance = [
    [0, 9, 14, 21, 23, 22, 25, 32, 36, 38, 42, 50, 52],
    [9, 0, 5, 12, 22, 21, 24, 31, 35, 37, 41, 49, 51],
    [14, 5, 0, 7, 17, 16, 23, 26, 30, 36, 36, 44, 46],
    [21, 12, 7, 0, 10, 21, 30, 27, 37, 43, 31, 37, 39],
    [23, 22, 17, 10, 0, 19, 28, 25, 35, 41, 29, 31, 29],
    [22, 21, 16, 21, 19, 0, 9, 10, 16, 22, 20, 28, 30],
    [25, 24, 23, 30, 28, 9, 0, 7, 11, 13, 17, 25, 27],
    [32, 31, 26, 27, 25, 10, 7, 0, 10, 16, 10, 18, 20],
    [36, 35, 30, 37, 35, 16, 11, 10, 0, 6, 6, 14, 16],
    [38, 37, 36, 43, 41, 22, 13, 16, 6, 0, 12, 12, 20],
    [42, 41, 36, 31, 29, 20, 17, 10, 6, 12, 0, 17, 10],
    [50, 49, 44, 37, 31, 28, 25, 18, 14, 12, 17, 0, 10],
    [52, 51, 46, 39, 29, 30, 27, 20, 16, 20, 10, 10, 0],
]

routes = []
for start in sorted(j for i, j in selected_arcs if i == 0):
    route = [0, start]
    while route[-1] != 0:
        route.append(successor[route[-1]])
    routes.append(route)

visited = [node for route in routes for node in route[1:-1]]
recomputed_total = 0
for index, route in enumerate(routes, start=1):
    route_demand = sum(demands[node] for node in route[1:-1])
    route_distance = sum(distance[i][j] for i, j in zip(route, route[1:]))
    recomputed_total += route_distance
    route_text = " -> ".join(f"P{node}" for node in route)
    print(f"Route {index}: {route_text}")
    print(f"         Demand: {route_demand:4d} | Distance: {route_distance}")
    assert route_demand <= 6000

arc_values = [
    value
    for variable_name, value in results["solution"].items()
    if parse_arc(variable_name) is not None
]
assert sorted(visited) == list(range(1, 13))
assert len(visited) == len(set(visited))
assert all(abs(value - round(value)) < 1e-6 for value in arc_values)
assert abs(recomputed_total - results["objective_value"]) < 1e-6
print(f"\nValidated total mileage: {recomputed_total}")

Route 1: P0 -> P4 -> P3 -> P2 -> P1 -> P0
         Demand: 5800 | Distance: 54
Route 2: P0 -> P5 -> P0
         Demand: 1700 | Distance: 44
Route 3: P0 -> P8 -> P9 -> P6 -> P0
         Demand: 5100 | Distance: 80
Route 4: P0 -> P10 -> P12 -> P11 -> P7 -> P0
         Demand: 5600 | Distance: 112

Validated total mileage: 290


## Comparison with the 1959 Paper

| Method | Distance | Interpretation |
|---|---:|---|
| Paper's two-stage aggregation | 294 | Near-optimal solution reported by Dantzig and Ramser |
| Paper's conjecture | 290 | Believed optimal, but not proved in the paper |
| Python exhaustive set partitioning | 290 | Exact enumeration for this finite instance |
| OPL CVRP solved by HiGHS | 290 | Proven optimal with zero MIP gap |

The two exact methods independently establish the paper's conjectured objective. Equivalent optimal tours may be reversed, and multiple route allocations may tie at the same objective; agreement is therefore based on feasibility and total mileage rather than one required arc orientation.

## References

- G. B. Dantzig and J. H. Ramser, "The Truck Dispatching Problem," *Management Science*, 6(1), 80-91, 1959. [JSTOR](https://www.jstor.org/stable/2627477)
- [Python reproduction](dantzig_ramser_truck_dispatching_problem.py)
- [OPL model](truck-dispatching-problem.mod)
- [OPL data](truck-dispatching-problem.dat)